# 10. Four ways to estimate a covariance matrix, and how to tell which is better

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same period |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | in the extension, the index the portfolio tracks, the cap-weighted combination of the 49 industries built in notebook 08 |
| Variance | the average squared distance of a series from its own average; its square root is the standard deviation, the usual measure of how much a return moves; volatility is the standard deviation of returns, stated per year here by multiplying the monthly figure by the square root of 12 |
| Covariance | a number that says how two series move together, positive when both tend to be above their averages in the same periods |
| Covariance matrix | the table of all variances and covariances of a set of assets, 49 by 49 here, 1,225 distinct numbers |
| Covariance estimator | a method for estimating the covariance matrix from a window of data |
| Estimation window | the past returns an estimator sees, 120 months or three years of trading days here |
| Estimation error | the difference between a number estimated from a window and its true value |
| Shrinkage | pulling an estimate part of the way towards a simpler target, which trades a little bias for less estimation error; the shrinkage intensity is the fraction of the way it is pulled |
| Factor | a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors are long-short portfolios built to isolate one such source each |
| Beta | how much an asset moves with a factor on average |
| Idiosyncratic | an asset's own movement, unrelated to the factors |
| Principal component | a combination of the assets, with one weight per asset, chosen so that it explains as much of the assets' total variance as a single combination can (notebook 09); its variance is called an eigenvalue of the covariance matrix |
| Parallel analysis | the comparison of each component's variance with what independent noise would produce, used in notebook 09 to count the components that are more than noise |
| Minimum-variance portfolio | the fully invested portfolio with the lowest variance under a given covariance matrix; long-only when no weight may be negative, unconstrained when weights may be negative |
| Turnover | the fraction of the portfolio bought and sold at a rebalance, summed over the assets; a turnover of 0.76 means that 76% of the portfolio changes hands in the month |
| Basis point | one hundredth of a percentage point, so 50 basis points is 0.50% |
| Sharpe ratio | the average monthly excess return divided by the standard deviation of the excess return; the reward earned per unit of risk taken; stated per year here as the annual excess return over the annual volatility |
| Active weight | the portfolio's weight in an asset minus the benchmark's weight in it |
| Tracking error | the standard deviation of the difference between two return series, stated per year |
| Bias statistic | the standard deviation of realised return divided by forecast standard deviation, over many periods; 1 when the forecasts of risk are right on average, above 1 when risk is under-forecast, below 1 when over-forecast |
| Condition number | the largest eigenvalue of a matrix divided by the smallest; it says how much inverting the matrix amplifies errors in it |
| Autocorrelation | the correlation of a series with its own value one period earlier |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |

**The problem, stated the way practitioners state it.** The key ingredient of every risk-based portfolio is the covariance matrix of the assets' returns. The natural estimate is the sample covariance, and it fails when the assets are many relative to the observations: for 49 industries and 120 monthly returns it has fewer than five observations per number. The literature answers with a long list of alternative estimators built on three ideas, shrinkage, time dynamics and factor structure, and it tests them almost always in one way, by the realised volatility of the unconstrained minimum-variance portfolio built from each. Dom, Howard, Jansen and Lohre (2024), whose introduction this paragraph follows, point out that such a portfolio is one no practitioner holds: it is leveraged, concentrated and trades a large part of its value every month, so the test rewards an estimator for what it does in positions nobody takes. They propose to judge estimators instead on realistic portfolios, long-only, weight-constrained and charged for trading, after costs, and they find that under those constraints the differences between estimators shrink, that time dynamics (recent days weighted more, as in RiskMetrics) matter more than shrinkage or structure, and that a simple dynamic estimator does as well as a complex one. This notebook takes their perspective. It builds four estimators at two data frequencies, judges each by the calibration of its risk forecasts and by both the unconstrained and the long-only minimum-variance portfolio it produces, and, in a section added after the first run, charges the portfolios for their trading, which is where the unconstrained portfolio's lower volatility turns out to be bought at a price no investor pays.

**What this notebook does.** The paper estimated the covariance matrix of its assets by the sample covariance of 120 monthly returns, and notebooks 02, 03 and 09 showed what that costs for 49 assets: 1,225 numbers from fewer than five observations each. This notebook builds four estimators and judges them on the 49 industries. The sample covariance uses every number the window offers. Ledoit-Wolf shrinkage pulls the sample covariance part of the way towards a simple target. The factor model describes the covariance by each industry's betas to six Fama-French factors plus its own variance. The principal-component model takes the market as its first factor and finds four more components in what the market leaves. Each is built at two data frequencies: the paper's 120 monthly returns, and three years of daily returns scaled to monthly, which is what practitioners use (Dom, Howard, Jansen and Lohre, 2024, estimate from daily returns over three years and rebalance monthly). Every estimate is built on the data before a month and judged on that month's realised returns, for every month from 1979-07 to the vintage's last month.

**How a covariance estimator is judged.** A covariance matrix makes two kinds of promise. The first is a forecast of risk: for any portfolio with weights w, w' Sigma w is the variance it predicts for next month. The bias statistic tests the promise over many months: divide each month's realised return by the forecast standard deviation, and take the standard deviation of those ratios; a calibrated forecaster gets 1, one that under-forecasts risk gets more than 1. The notebook tests it on each of the 49 industries, on 1/N, on the benchmark, and on 200 random active portfolios (weights that sum to zero, largest 2 percentage points, the extension's active weight bound), which is the tracking-error forecast the extension will rely on. The second promise is in the inverse: the minimum-variance portfolio uses Sigma^-1, and errors in the small directions of Sigma blow up there. Following Dom, Howard, Jansen and Lohre (2024), the notebook builds the minimum-variance portfolio from each estimate every month, with and without the long-only constraint, holds it for a month, and reports the volatility it realised over all months; a better covariance gives a lower one. The condition number of each estimate says how much its inverse amplifies errors.

**What I expect to see, written before the run (`constants.py`, 3 October 2026).**

1. Count first: the daily panel from 1969-07 has one missing day (Software on 1971-03-11), the daily factor files cover every day of every window, and the evaluation months are counted and the same for every estimator.
2. The frequency question: for the sample estimator, three years of daily returns give a bias statistic closer to 1 than 120 monthly returns for the benchmark and for the active portfolios, and a lower realised volatility of the unconstrained minimum-variance portfolio.
3. The finding of Dom, Howard, Jansen and Lohre: with daily data, the long-only minimum-variance portfolios of the four estimators realise volatilities within one percentage point a year of each other, and the unconstrained portfolio of the sample estimator on monthly data realises the highest volatility of all.
4. The scaling check: the benchmark's realised monthly variance over a three-year window divided by 21 times its daily variance over the same window averages between 0.8 and 1.25 over the windows; outside that, the daily estimators are still used as designed and the notebook says by how much and in which direction the scaling misses.
5. Reported with no pass or fail: the shrinkage intensity over time at both frequencies, the sensitivities (one- and five-year daily windows, exponential weighting with a one-year half-life, the constant-correlation shrinkage target, a two-component model), and the condition numbers.

Running time: about one minute. Three further tests, P1 to P3, were added after the run and are labelled as such below; their expectations were fixed in `constants.py` on 4 October 2026 before their code was written.

## Modules

`constants`, `french_loader`, `benchmark` and `rules` are those of earlier notebooks; `rules` supplies the exact long-only minimum-variance solver of notebook 03. `covariance` is new: the four estimators, the sensitivities, the windows and the tests, each with a test on a made-up market with a known covariance in `tests/test_covariance.py`.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Attribution (notebook 13): factor-based attribution of each constrained
# portfolio's realised active return against the factors above, contribution
# by factor plus residual, with the identity as the check.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # contributions plus residual minus active return, per month, in return units

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

In [ ]:
%%writefile src/bp/covariance.py
"""Four ways to estimate a covariance matrix from a window of excess returns, and the tests that judge them (notebook 10).

Every estimator takes a window of excess returns (rows are periods, columns are
assets) and returns an N by N covariance matrix for the period that follows. The
sample covariance uses all N(N+1)/2 numbers the window offers. Ledoit-Wolf
shrinkage pulls it towards a simple target. The factor model and the
principal-component model describe it with a small number of common sources of movement
plus each asset's own variance. The tests measure how well each estimate
forecasts the variance of test portfolios in the next month (the bias
statistic) and how well the minimum-variance portfolio built from it does
(the tests of Dom, Howard, Jansen and Lohre, 2024).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import rules as S

PERIODS_PER_YEAR = 12


# ---------------------------------------------------------------------------
# Estimators
# ---------------------------------------------------------------------------

def sample_cov(X: np.ndarray) -> np.ndarray:
    """The sample covariance, denominator T - 1."""
    return np.cov(np.asarray(X, dtype=float), rowvar=False, ddof=1)


def ewma_cov(X: np.ndarray, halflife: float) -> np.ndarray:
    """Exponentially weighted covariance: the weight of an observation halves every `halflife` periods, the most recent weighing most.

    The mean is weighted the same way; weights are normalised to sum to one.
    """
    X = np.asarray(X, dtype=float)
    T = X.shape[0]
    lam = 0.5 ** (1.0 / halflife)
    w = lam ** np.arange(T - 1, -1, -1)
    w = w / w.sum()
    mu = w @ X
    D = X - mu
    return (D * w[:, None]).T @ D / (1.0 - (w ** 2).sum())   # the unbiased normalisation for weighted samples


def lw_scaled_identity(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004b): (1 - k) S + k m I, with S the sample covariance (denominator T), m the average
    sample variance and k the intensity that minimises the expected squared error.

    Returns the estimate and k. The intensity is b^2 / d^2 clipped to [0, 1], where d^2 is the squared
    distance between S and m I and b^2 measures how much S moves from one observation to the next.
    """
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    m = np.trace(S) / N
    d2 = ((S - m * np.eye(N)) ** 2).sum() / N
    b2_bar = sum(((np.outer(x, x) - S) ** 2).sum() for x in D) / (N * T * T)
    b2 = min(b2_bar, d2)
    k = float(b2 / d2) if d2 > 0 else 0.0
    return (1.0 - k) * S + k * m * np.eye(N), k


def lw_constant_correlation(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004a): shrink S towards the matrix with the same variances and one common correlation,
    the average sample correlation; the intensity follows the paper's appendix. Returns the estimate and k."""
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    var = np.diag(S)
    sd = np.sqrt(var)
    corr = S / np.outer(sd, sd)
    rbar = (corr.sum() - N) / (N * (N - 1))
    F = rbar * np.outer(sd, sd)
    np.fill_diagonal(F, var)
    # pi: the sum over i, j of the estimated asymptotic variances of the entries of sqrt(T) S
    Y = D ** 2
    pi_mat = (Y.T @ Y) / T - S ** 2
    pi = pi_mat.sum()
    # rho: the diagonal part plus the covariance between the entries of S and the target's correlation structure
    term1 = ((D ** 3).T @ D) / T          # theta_{ii,ij}: average over t of x_it^3 x_jt
    help_ = term1 - var[:, None] * S      # (1/T) sum_t (x_it^2 - S_ii)(x_it x_jt - S_ij)
    theta_ii = help_
    theta_jj = help_.T
    rho_off = (rbar / 2.0) * (np.sqrt(var[None, :] / var[:, None]) * theta_ii + np.sqrt(var[:, None] / var[None, :]) * theta_jj)
    np.fill_diagonal(rho_off, 0.0)
    rho = np.trace(pi_mat) + rho_off.sum()
    gamma = ((F - S) ** 2).sum()
    k = (pi - rho) / gamma / T if gamma > 0 else 0.0
    k = float(min(1.0, max(0.0, k)))
    return (1.0 - k) * S + k * F, k


def factor_cov(X: np.ndarray, F: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The exact factor model: X regressed on the factors F (with an intercept) over the window;
    the estimate is B Sigma_F B' plus the diagonal of residual variances. Returns the estimate and B (N by K)."""
    X = np.asarray(X, dtype=float)
    F = np.asarray(F, dtype=float)
    T, K = F.shape
    A = np.column_stack([np.ones(T), F])
    coef, *_ = np.linalg.lstsq(A, X, rcond=None)      # (K + 1) by N
    B = coef[1:].T                                     # N by K
    resid = X - A @ coef
    Sigma_F = np.cov(F, rowvar=False, ddof=1).reshape(K, K)
    resid_var = resid.var(axis=0, ddof=K + 1)
    return B @ Sigma_F @ B.T + np.diag(resid_var), B


def pca_cov(X: np.ndarray, n_components: int, market: np.ndarray | None = None) -> tuple[np.ndarray, int]:
    """The principal-component model.

    With `market` given (constants.PCA_FIRST_COMPONENT == "market"): each asset is regressed on the market
    series, and the covariance of the residuals is described by its first n_components - 1 principal
    components plus the diagonal of what they leave; the estimate is beta beta' var(market) plus that.
    Without a market series, the first n_components of X's own covariance plus the diagonal remainder.
    Returns the estimate and the number of components taken from the data.
    """
    X = np.asarray(X, dtype=float)
    if market is not None:
        m = np.asarray(market, dtype=float)
        A = np.column_stack([np.ones(len(m)), m])
        coef, *_ = np.linalg.lstsq(A, X, rcond=None)
        beta = coef[1]
        resid = X - A @ coef
        base = np.outer(beta, beta) * m.var(ddof=1)
        k = n_components - 1
        target = np.cov(resid, rowvar=False, ddof=2)
    else:
        base = 0.0
        k = n_components
        target = sample_cov(X)
    if k <= 0:
        return base + np.diag(np.diag(target)), 0
    vals, vecs = np.linalg.eigh(target)
    order = np.argsort(vals)[::-1][:k]
    V = vecs[:, order]
    L = np.clip(vals[order], 0.0, None)
    common = (V * L) @ V.T
    own = np.diag(np.clip(np.diag(target) - np.diag(common), 1e-12, None))
    return base + common + own, k


def scale_daily_to_monthly(Sigma_daily: np.ndarray, scale: float = C.DAILY_TO_MONTHLY_SCALE) -> np.ndarray:
    """A daily covariance times the number of trading days in a month; exact when daily returns do not predict the next day's."""
    return Sigma_daily * scale


def variance_ratio(x: np.ndarray, q: int = C.COV_SCALE_VR_HORIZON_DAYS) -> float:
    """The variance ratio of Lo and MacKinlay (1988): the variance of the overlapping sums of q consecutive
    values divided by q times the variance of the single values. It is 1 when the values do not predict each
    other, above 1 when a high value tends to be followed by a high one (positive autocorrelation) and below 1
    when a high value tends to be followed by a low one. Multiplying a daily covariance by q times this ratio,
    in place of q alone, gives the covariance of q-day sums that the autocorrelation implies."""
    v = np.asarray(x, dtype=float)
    if len(v) < 2 * q:
        return float("nan")
    sums = np.convolve(v, np.ones(q), mode="valid")
    return float(sums.var(ddof=1) / (q * v.var(ddof=1)))


# ---------------------------------------------------------------------------
# Windows
# ---------------------------------------------------------------------------

def monthly_window(excess_m: pd.DataFrame, month: pd.Period, months: int = C.EXT_ESTIMATION_WINDOW) -> pd.DataFrame:
    """The `months` monthly rows before `month`."""
    end = month - 1
    start = month - months
    return excess_m.loc[start:end]


def daily_window(excess_d: pd.DataFrame, month: pd.Period, years: int) -> pd.DataFrame:
    """The trading days of the 12 * years calendar months before `month`, days with any missing value left out."""
    end = (month - 1).asfreq("D", "end")
    start = (month - 12 * years).asfreq("D", "start")
    return excess_d.loc[start:end].dropna()


def daily_benchmark_excess(returns_d: pd.DataFrame, rf_d: pd.Series, weights_m: pd.DataFrame) -> pd.Series:
    """The benchmark's daily excess return: each day's industry returns weighted by that month's benchmark weights, minus the daily risk-free rate."""
    month_of_day = returns_d.index.asfreq("M")
    w = weights_m.reindex(month_of_day)
    w.index = returns_d.index
    return ((w * returns_d).sum(axis=1, min_count=returns_d.shape[1]) - rf_d.reindex(returns_d.index)).rename("benchmark_excess")


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def bias_statistic(realised: np.ndarray, predicted_var: np.ndarray) -> float:
    """The standard deviation of realised / sqrt(predicted variance): 1 when the forecasts are calibrated, above 1 when risk is under-forecast."""
    z = np.asarray(realised, dtype=float) / np.sqrt(np.asarray(predicted_var, dtype=float))
    return float(z.std(ddof=1))


def random_active_weights(n_assets: int, draws: int = C.COV_TEST_ACTIVE_DRAWS, bound: float = C.ACTIVE_WEIGHT_BOUND,
                          seed: int = C.COV_TEST_SEED) -> np.ndarray:
    """`draws` active-weight vectors: weights sum to zero and the largest absolute weight equals `bound`."""
    rng = np.random.default_rng(seed)
    U = rng.uniform(-1.0, 1.0, size=(draws, n_assets))
    U = U - U.mean(axis=1, keepdims=True)
    return bound * U / np.abs(U).max(axis=1, keepdims=True)


def min_variance_unconstrained(Sigma: np.ndarray) -> np.ndarray:
    """w proportional to Sigma^-1 1, scaled to sum to one; negative weights allowed."""
    x = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return x / x.sum()


def min_variance_long_only(Sigma: np.ndarray) -> np.ndarray:
    """The long-only minimum-variance portfolio, solved exactly by non-negative least squares (rules.constrained_min_variance)."""
    return S.constrained_min_variance(Sigma, 0.0)


def condition_number(Sigma: np.ndarray) -> float:
    vals = np.linalg.eigvalsh(Sigma)
    return float(vals.max() / max(vals.min(), 1e-300))


def annualised_vol(x: pd.Series | np.ndarray) -> float:
    return float(np.std(np.asarray(x, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


# ---------------------------------------------------------------------------
# The panels and the estimator variants, as notebooks 10 to 12 use them
# ---------------------------------------------------------------------------

class Panels:
    """Everything the estimators read: monthly and daily excess returns of the 49 industries, the six factors at both
    frequencies, the benchmark's weights and its excess return at both frequencies, and the plain monthly returns."""

    def __init__(self, start: str = C.EXT_SAMPLE_START):
        from . import french_loader as fl
        from . import benchmark as BM
        inputs = BM.load_inputs()
        f3 = fl.load_monthly("factors3", 0)
        f5 = fl.load_monthly("factors5", 0)
        mom = fl.load_monthly("momentum", 0)
        start_p = pd.Period(start, "M")
        self.returns_m = inputs["returns"].loc[start_p:]
        self.last = self.returns_m.index[-1]
        self.rf_m = f3["RF"].loc[start_p:self.last]
        self.excess_m = self.returns_m.sub(self.rf_m, axis=0)
        self.weights = BM.cap_weights(inputs["firms"].loc[start_p:], inputs["size"].loc[start_p:], C.CAPW_WEIGHT_TIMING)
        self.bench_m = (BM.combination_return(self.returns_m, self.weights) - self.rf_m).rename("benchmark_excess")
        self.factors_m = pd.concat([f5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1).loc[start_p:self.last]
        returns_d = fl.load_monthly("ind49_daily", r"Value Weighted Returns -- Daily").loc[start_p.asfreq("D", "start"):]
        f3d = fl.load_monthly("factors3_daily", 0)
        f5d = fl.load_monthly("factors5_daily", 0)
        momd = fl.load_monthly("momentum_daily", 0)
        self.rf_d = f3d["RF"].reindex(returns_d.index)
        self.returns_d = returns_d
        self.excess_d = returns_d.sub(self.rf_d, axis=0)
        self.factors_d = pd.concat([f5d[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], momd["Mom"]], axis=1).reindex(returns_d.index)
        self.bench_d = daily_benchmark_excess(returns_d, self.rf_d, self.weights)
        self.files = ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")


DAILY_YEARS = {"daily_3y": C.EXT_COV_DAILY_WINDOW_YEARS, "daily_1y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[0],
               "daily_5y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1], "daily_5y_ewma": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1]}


def estimate(name: str, freq: str, month: pd.Period, P: Panels) -> tuple[np.ndarray, float, int]:
    """One estimator variant for `month`, built on the data before it: the monthly covariance (daily ones scaled by
    DAILY_TO_MONTHLY_SCALE), the shrinkage intensity (nan for the others) and the number of observations in the window.
    The same dispatch as notebook 10's evaluation loop."""
    if freq == "monthly_120":
        X = monthly_window(P.excess_m, month); F = P.factors_m.loc[X.index]; mk = P.bench_m.loc[X.index]; scale = 1.0
    else:
        X = daily_window(P.excess_d, month, DAILY_YEARS[freq]); F = P.factors_d.loc[X.index]; mk = P.bench_d.loc[X.index]; scale = C.DAILY_TO_MONTHLY_SCALE
    Xv = X.to_numpy(); intensity = np.nan
    if name == "sample":
        S = ewma_cov(Xv, C.EXT_COV_EWMA_HALFLIFE_DAYS) if freq.endswith("ewma") else sample_cov(Xv)
    elif name == "ledoit_wolf":
        S, intensity = lw_scaled_identity(Xv)
    elif name == "ledoit_wolf_cc":
        S, intensity = lw_constant_correlation(Xv)
    elif name == "factor":
        S, _ = factor_cov(Xv, F.to_numpy())
    elif name == "pca":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS, market=mk.to_numpy())
    elif name == "pca_2":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS_SENSITIVITY, market=mk.to_numpy())
    else:
        raise KeyError(name)
    return S * scale, intensity, len(X)


# The five estimator variants of the rolling evaluation (notebook 12): the four estimators on three years of daily
# returns and the paper's convention, the sample covariance of 120 monthly returns.
EVALUATION_VARIANTS = tuple((e, "daily_3y") for e in C.COV_ESTIMATORS) + (("sample", "monthly_120"),)
# The robust portfolio (constants.ROBUST_VARIANT) is built from all five covariances at once, and RiskMetrics
# (constants.EWMA_VARIANT) is the recency-weighted sample covariance; the seven together are the variants the rolling evaluation reports.
ALL_VARIANTS = EVALUATION_VARIANTS + (C.ROBUST_VARIANT, C.EWMA_VARIANT)

## Count first: monthly and daily panels, factors and the benchmark at both frequencies

The 49 industries' excess returns monthly from 1969-07 and daily from 1969-07-01; the six factors (Mkt-RF, SMB, HML, RMW, CMA, Mom) at both frequencies; the benchmark of notebook 08 monthly, and daily with each month's weights applied to that month's days.

In [ ]:
import sys
sys.path.insert(0, "src")
import time
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import benchmark as BM
from bp import covariance as CV

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

inputs = BM.load_inputs()
f3 = fl.load_monthly("factors3", 0)
f5 = fl.load_monthly("factors5", 0)
mom = fl.load_monthly("momentum", 0)
start = pd.Period(C.EXT_SAMPLE_START, "M")
returns_m = inputs["returns"].loc[start:]
LAST = returns_m.index[-1]
rf_m = f3["RF"].loc[start:LAST]
excess_m = returns_m.sub(rf_m, axis=0)
weights = BM.cap_weights(inputs["firms"].loc[start:], inputs["size"].loc[start:], C.CAPW_WEIGHT_TIMING)
bench_m = (BM.combination_return(returns_m, weights) - rf_m).rename("benchmark_excess")
factors_m = pd.concat([f5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1).loc[start:LAST]

returns_d = fl.load_monthly("ind49_daily", r"Value Weighted Returns -- Daily").loc[start.asfreq("D", "start"):]
f3d = fl.load_monthly("factors3_daily", 0)
f5d = fl.load_monthly("factors5_daily", 0)
momd = fl.load_monthly("momentum_daily", 0)
rf_d = f3d["RF"].reindex(returns_d.index)
excess_d = returns_d.sub(rf_d, axis=0)
factors_d = pd.concat([f5d[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], momd["Mom"]], axis=1).reindex(returns_d.index)
bench_d = CV.daily_benchmark_excess(returns_d, rf_d, weights)

eval_months = pd.period_range(C.COV_EVAL_START, LAST, freq="M")
first_daily_day = (eval_months[0] - 12 * max(C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY + (C.EXT_COV_DAILY_WINDOW_YEARS,))).asfreq("D", "start")

# Expectation 1
assert excess_m.shape == (len(pd.period_range(start, LAST, freq="M")), 49) and not excess_m.isna().any().any()
missing_days = excess_d.index[excess_d.isna().any(axis=1)]
assert len(missing_days) == 1 and str(missing_days[0]) == "1971-03-11" and excess_d.loc[missing_days[0]].isna().sum() == 1
assert rf_d.notna().all(), "the daily risk-free rate has a gap"
assert not factors_d.loc[first_daily_day:].isna().any().any(), "a daily factor is missing inside the daily windows"
assert not factors_m.loc[eval_months[0] - C.EXT_ESTIMATION_WINDOW:].isna().any().any()
assert bench_m.loc[eval_months].notna().all() and bench_d.loc[first_daily_day:].notna().sum() == excess_d.loc[first_daily_day:].dropna().shape[0]
provenance = [fl.download(key) for key in ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")]
fl.write_provenance(provenance)
print(f"monthly panel: {len(excess_m)} months, {start} to {LAST}; daily panel: {len(excess_d):,} trading days, {excess_d.index[0]} to {excess_d.index[-1]}, "
      f"{len(missing_days)} day with a missing industry return ({missing_days[0]}, {excess_d.columns[excess_d.loc[missing_days[0]].isna()][0]}), left out of every window")
print(f"evaluation months: {len(eval_months)}, {eval_months[0]} to {eval_months[-1]}; the 120-month window of the first ends {eval_months[0] - 1}, its three-year daily window starts {(eval_months[0] - 36).asfreq('D', 'start')}")
print(f"a three-year daily window holds about {int(round(len(excess_d.loc['2000-01-01':'2002-12-31']) ))} trading days (2000 to 2002), {int(round(len(excess_d.loc['2000-01-01':'2002-12-31']) * 49))} observations for 1,225 numbers; the monthly window holds {120 * 49:,}")
print("expectation 1 holds; CRSP vintage of the eight files:", ", ".join(sorted({r["vintage_line"].split("the ")[1].split(" ")[0] for r in provenance})))

## The four estimators, in words

- **Sample covariance.** Each of the 1,225 numbers is the average over the window of the product of two industries' deviations from their means. It is unbiased and it uses every number; with few observations per number it is noisy, and its smallest directions, the combinations of industries that happened to move least in the window, are the noisiest of all.

- **Ledoit-Wolf shrinkage.** The estimate is (1 - k) times the sample covariance plus k times a target matrix that has the same average variance on its diagonal and zero everywhere else (the scaled identity). The intensity k is computed from the data by the formula of Ledoit and Wolf (2004b): the noisier the sample covariance relative to its distance from the target, the larger k. Shrinkage lifts the smallest directions and lowers the largest, so the inverse is tamer. The target with the same variances and one common correlation (Ledoit and Wolf 2004a) is run beside it; Dom, Howard, Jansen and Lohre (2024) found it inferior, which is why the scaled identity is the design.

- **Factor model.** Each industry's excess return is regressed, over the window, on six factors: the market, size, value, profitability, investment and momentum factors of Fama and French. The covariance is then the betas times the factors' covariance times the betas, plus each industry's residual variance on the diagonal. The 1,225 numbers become 49 times 6 betas, a 6 by 6 factor covariance and 49 residual variances, 364 numbers, each estimated from more information; the price is the assumption that whatever two industries share beyond the six factors is zero.

- **Principal-component model.** Notebook 09 found that the first component of the 49 industries is the market and that it parts from the cap-weighted benchmark in concentrated markets, so the design (`PCA_FIRST_COMPONENT`) imposes the market: each industry is regressed on the benchmark's excess return, and the covariance of the residuals is described by its first four principal components plus the diagonal of what they leave. The estimate is the betas times the benchmark's variance times the betas, plus that residual model. A version with one residual component (the count parallel analysis found above noise in notebook 09) is run beside it.

- **Two frequencies.** Each estimator sees either the last 120 monthly excess returns or the trading days of the last 36 calendar months, about 756 of them. A daily covariance is multiplied by 21, the number of trading days in an average month, to make it a monthly covariance; the step is exact when one day's return says nothing about the next day's, and expectation 4 checks it.

## The evaluation loop

For every month from 1979-07, each estimator is built on the data before the month, and four things are recorded:

- the variance it predicts for each test portfolio, beside the portfolio's realised excess return in the month;
- the long-only and the unconstrained minimum-variance portfolios built from it, their realised returns in the month and their change of weights from the previous month;
- the condition number;
- for the shrinkage estimators, the intensity.

In [ ]:
A = CV.random_active_weights(49)
EW = np.ones(49) / 49
MAIN = [(e, f) for f in C.EXT_COV_FREQUENCIES for e in C.COV_ESTIMATORS]
SENSITIVITY = [("sample", "daily_1y"), ("sample", "daily_5y"), ("sample", "daily_5y_ewma"),
               ("ledoit_wolf_cc", "monthly_120"), ("ledoit_wolf_cc", "daily_3y"), ("pca_2", "monthly_120"), ("pca_2", "daily_3y")]
VARIANTS = MAIN + SENSITIVITY
YEARS = {"daily_3y": C.EXT_COV_DAILY_WINDOW_YEARS, "daily_1y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[0],
         "daily_5y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1], "daily_5y_ewma": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1]}


def estimate(name, freq, month):
    if freq == "monthly_120":
        X = CV.monthly_window(excess_m, month); F = factors_m.loc[X.index]; mk = bench_m.loc[X.index]; scale = 1.0
    else:
        X = CV.daily_window(excess_d, month, YEARS[freq]); F = factors_d.loc[X.index]; mk = bench_d.loc[X.index]; scale = C.DAILY_TO_MONTHLY_SCALE
    Xv = X.to_numpy(); intensity = np.nan
    if name == "sample":
        S = CV.ewma_cov(Xv, C.EXT_COV_EWMA_HALFLIFE_DAYS) if freq.endswith("ewma") else CV.sample_cov(Xv)
    elif name == "ledoit_wolf":
        S, intensity = CV.lw_scaled_identity(Xv)
    elif name == "ledoit_wolf_cc":
        S, intensity = CV.lw_constant_correlation(Xv)
    elif name == "factor":
        S, _ = CV.factor_cov(Xv, F.to_numpy())
    elif name == "pca":
        S, _ = CV.pca_cov(Xv, C.PCA_N_COMPONENTS, market=mk.to_numpy())
    elif name == "pca_2":
        S, _ = CV.pca_cov(Xv, C.PCA_N_COMPONENTS_SENSITIVITY, market=mk.to_numpy())
    return S * scale, intensity, len(X)


collected = {v: [] for v in VARIANTS}
previous = {v: (None, None) for v in VARIANTS}
t0 = time.time()
for i, month in enumerate(eval_months):
    r = excess_m.loc[month].to_numpy()
    b = weights.loc[month].to_numpy()
    for v in VARIANTS:
        S, intensity, n_obs = estimate(*v, month)
        w_lo = CV.min_variance_long_only(S)
        w_un = CV.min_variance_unconstrained(S)
        p_lo, p_un = previous[v]
        collected[v].append({
            "month": month, "n_obs": n_obs, "intensity": intensity, "condition": CV.condition_number(S),
            "pred_industries": np.diag(S), "pred_ew": EW @ S @ EW, "pred_bench": b @ S @ b, "pred_active": np.einsum("ij,jk,ik->i", A, S, A),
            "real_industries": r, "real_ew": EW @ r, "real_bench": b @ r, "real_active": A @ r,
            "gmv_lo_ret": w_lo @ r, "gmv_un_ret": w_un @ r,
            "gmv_lo_turnover": np.abs(w_lo - p_lo).sum() if p_lo is not None else np.nan,
            "gmv_un_turnover": np.abs(w_un - p_un).sum() if p_un is not None else np.nan,
            "gmv_lo_held": int((w_lo > 1e-6).sum()), "gmv_un_max_abs": float(np.abs(w_un).max()),
        })
        previous[v] = (w_lo, w_un)
    if i % 100 == 0:
        print(f"{month}: {time.time() - t0:.0f}s")
frames = {v: pd.DataFrame(recs).set_index("month") for v, recs in collected.items()}
print(f"{len(eval_months)} months, {len(VARIANTS)} estimator variants, {time.time() - t0:.0f}s")

## Expectations 2 and 3: the forecasts of risk and the minimum-variance portfolios

One row per estimator and data frequency. Bias statistics first (1 is calibrated), then the realised volatility of the long-only and the unconstrained minimum-variance portfolios in percent a year, their average monthly turnover, how many of the 49 industries the long-only portfolio holds, and the median condition number.

In [ ]:
def summarise(v):
    df = frames[v]
    ind = [CV.bias_statistic(np.stack(df.real_industries)[:, j], np.stack(df.pred_industries)[:, j]) for j in range(49)]
    act = [CV.bias_statistic(np.stack(df.real_active)[:, j], np.stack(df.pred_active)[:, j]) for j in range(A.shape[0])]
    return {"estimator": v[0], "data": v[1], "observations": int(df.n_obs.median()),
            "bias: benchmark": CV.bias_statistic(df.real_bench, df.pred_bench), "bias: 1/N": CV.bias_statistic(df.real_ew, df.pred_ew),
            "bias: industries, mean": float(np.mean(ind)), "bias: industries, range": f"{min(ind):.2f} to {max(ind):.2f}",
            "bias: active, mean": float(np.mean(act)),
            "GMV long-only vol %": 100 * CV.annualised_vol(df.gmv_lo_ret), "GMV unconstrained vol %": 100 * CV.annualised_vol(df.gmv_un_ret),
            "turnover long-only": df.gmv_lo_turnover.mean(), "turnover unconstrained": df.gmv_un_turnover.mean(),
            "industries held (long-only)": df.gmv_lo_held.mean(), "largest |weight| (unconstrained, median)": df.gmv_un_max_abs.median(),
            "condition number (median)": df.condition.median(), "shrinkage intensity (median)": df.intensity.median()}

table = pd.DataFrame([summarise(v) for v in VARIANTS]).set_index(["estimator", "data"])
main = table.loc[[v for v in MAIN]]
print("the four estimators at the two frequencies:")
print(main.to_string(float_format=lambda x: f"{x:.3f}"))

s_m, s_d = table.loc[("sample", "monthly_120")], table.loc[("sample", "daily_3y")]
e2_bench = abs(s_d["bias: benchmark"] - 1) < abs(s_m["bias: benchmark"] - 1)
e2_active = abs(s_d["bias: active, mean"] - 1) < abs(s_m["bias: active, mean"] - 1)
e2_gmv = s_d["GMV unconstrained vol %"] < s_m["GMV unconstrained vol %"]
print()
print(f"expectation 2, the sample estimator on daily against monthly data: bias for the benchmark {s_d['bias: benchmark']:.3f} against {s_m['bias: benchmark']:.3f} "
      f"({'closer to 1' if e2_bench else 'NOT closer to 1'}); bias for the active portfolios {s_d['bias: active, mean']:.3f} against {s_m['bias: active, mean']:.3f} "
      f"({'closer to 1' if e2_active else 'NOT closer to 1'}); unconstrained minimum-variance volatility {s_d['GMV unconstrained vol %']:.1f}% against {s_m['GMV unconstrained vol %']:.1f}% "
      f"({'lower' if e2_gmv else 'NOT lower'}): {sum([e2_bench, e2_active, e2_gmv])} of 3 parts hold")
lo = main.xs("daily_3y", level="data")["GMV long-only vol %"]
e3_range = (lo.max() - lo.min()) / 100
un_all = table["GMV unconstrained vol %"]
e3_sample_highest_main = main["GMV unconstrained vol %"].idxmax() == ("sample", "monthly_120")
e3_sample_highest_all = un_all.idxmax() == ("sample", "monthly_120")
print(f"expectation 3: long-only minimum-variance volatility at daily_3y runs from {lo.min():.2f}% to {lo.max():.2f}%, a range of {100 * e3_range:.2f} points "
      f"(level {100 * C.COV_LONG_ONLY_RANGE_MAX:.0f} point): {'MET' if e3_range < C.COV_LONG_ONLY_RANGE_MAX else 'NOT MET'}; "
      f"the sample estimator on monthly data has the highest unconstrained volatility among the eight main variants: {'yes' if e3_sample_highest_main else 'no'} ({s_m['GMV unconstrained vol %']:.1f}%), "
      f"and among all {len(VARIANTS)} variants: {'yes' if e3_sample_highest_all else 'no'} (highest: {un_all.idxmax()}, {un_all.max():.1f}%)")

## Expectation 4: does 21 times a daily covariance equal a monthly covariance

For every three-year window, the benchmark's realised monthly variance divided by 21 times its daily variance over the same window. The ratio is 1 when one day's return says nothing about the next day's. Beside it, the autocorrelation of the benchmark's daily excess return in the window (the correlation between one day's return and the next day's): positive autocorrelation makes monthly variance larger than 21 times the daily variance, negative autocorrelation makes it smaller.

In [ ]:
rows = []
for month in eval_months:
    Xd = CV.daily_window(excess_d, month, C.EXT_COV_DAILY_WINDOW_YEARS)
    bd = bench_d.loc[Xd.index]
    bm = bench_m.loc[month - 12 * C.EXT_COV_DAILY_WINDOW_YEARS: month - 1]
    rows.append({"month": month, "ratio": bm.var(ddof=1) / (C.DAILY_TO_MONTHLY_SCALE * bd.var(ddof=1)), "daily autocorrelation": bd.autocorr(1)})
scaling = pd.DataFrame(rows).set_index("month")
mean_ratio = scaling["ratio"].mean()
lo_r, hi_r = C.COV_SCALE_RATIO_RANGE
print(f"expectation 4: mean ratio {mean_ratio:.3f}, median {scaling['ratio'].median():.3f} (range {lo_r} to {hi_r}): {'MET' if lo_r <= mean_ratio <= hi_r else 'NOT MET'}")
by_decade = scaling.groupby((scaling.index.year // 10) * 10).agg(["mean", "min", "max"])
by_decade.index = [f"windows ending in the {d}s" for d in by_decade.index]
print()
print("by decade of the window's last month:")
print(by_decade.to_string(float_format=lambda x: f"{x:.3f}"))
print()
print(f"windows with a ratio above {hi_r}: {int((scaling['ratio'] > hi_r).sum())} of {len(scaling)}; below {lo_r}: {int((scaling['ratio'] < lo_r).sum())}")
print(f"correlation across windows between the ratio and the daily autocorrelation: {scaling['ratio'].corr(scaling['daily autocorrelation']):.2f}")

## Expectation 5: the sensitivities, the shrinkage intensity over time, and the condition numbers

The sensitivities are the sample estimator on one and five years of daily returns and with exponential weighting (one-year half-life) over five years, Ledoit-Wolf shrinkage towards the constant-correlation target, and the principal-component model with one residual component. Then the shrinkage intensity by decade at both frequencies, and the condition numbers.

In [ ]:
sens = table.loc[[v for v in SENSITIVITY]]
print("sensitivities:")
print(sens.to_string(float_format=lambda x: f"{x:.3f}"))
print()
intensity = pd.DataFrame({f"{v[0]} {v[1]}": frames[v]["intensity"] for v in [("ledoit_wolf", "monthly_120"), ("ledoit_wolf", "daily_3y"), ("ledoit_wolf_cc", "monthly_120"), ("ledoit_wolf_cc", "daily_3y")]})
print("shrinkage intensity k (median by decade of the estimation month):")
print(intensity.groupby((intensity.index.year // 10) * 10).median().to_string(float_format=lambda x: f"{x:.3f}"))
print()
cond = pd.DataFrame({f"{v[0]} {v[1]}": frames[v]["condition"] for v in MAIN})
print("condition number (median by decade):")
print(cond.groupby((cond.index.year // 10) * 10).median().to_string(float_format=lambda x: f"{x:,.0f}"))

## Added after the run: three open points, tested

The first run of this notebook (3 October 2026) ended with a list of open points. Three of them can be tested with the data already collected, and were, on 4 October 2026, with the expectations fixed in `constants.py` before the code was written. The cells below are labelled as additions; nothing above them changed.

- **P1, a scale correction that uses the autocorrelation.** Expectation 4 found that 21 times a daily variance is not a monthly variance when one day's return predicts the next day's. The correction measures how far the days are from independent inside the window and scales by that. For every window, the variance ratio is the variance of the benchmark's 21-day excess returns (every run of 21 consecutive days in the window) divided by 21 times the variance of its daily excess returns. It is 1 when days are independent, above 1 under positive autocorrelation, below 1 under negative. The corrected forecast multiplies the daily covariance by the variance ratio times 21 in place of 21. One number multiplies every entry of the matrix, so the minimum-variance weights and everything in expectations 2 and 3 about them are unchanged; only the forecasts of variance move. The correction is one number taken from the benchmark and not a matrix of autocovariances, because the matrix version (the variance of a 21-day sum of all 49 daily returns, the Newey-West estimate with 20 lags) has about fourteen times the sampling variance of the sample covariance, which would leave the equivalent of about 54 daily observations per entry and give away what daily data are for. The variance ratio inside the window is close to the ratio of expectation 4 by construction, so the test is out of window: the bias statistics on the months that follow. Main version: the variance ratio over the estimator's own three-year window. Sensitivity: over ten years of daily benchmark returns, because the autocorrelation moves over decades and a longer window trades noise for lag. Expectation P1a: for the benchmark and the sample estimator, the bias statistic by decade under the correction is closer to 1 than under the 21 rule in at least four of the five decades. P1b, reported with no verdict: the bias statistics for 1/N, the industries and the active portfolios under the correction; a portfolio whose own autocorrelation differs from the benchmark's keeps part of its miss.
- **P2, bias statistics by decade.** A bias statistic over 566 months at once can be close to 1 for a forecaster that under-forecasts risk for thirty years and over-forecasts it for twenty-five. The decades are 1979-07 to 1989-12, then each calendar decade, then 2020-01 to the last month. Expectation P2a, derived from the ratio table of expectation 4: under the 21 rule the benchmark's bias statistic on daily data is above 1 in the first decade, below 1 in the 2010s, and higher in the first decade than in the 2000s.
- **P3, trading costs.** The minimum-variance tests charged no trading costs, and the unconstrained portfolios trade a large part of their value every month. Here each portfolio's cost is its average monthly turnover times the cost per unit of turnover times 12, at the two cost levels of the replication (50 basis points per unit of turnover, DGU's figure, and 100 as the sensitivity), beside its gross excess return (the mean monthly excess return times 12), its net excess return (gross minus cost), its realised volatility and its Sharpe ratio net of cost (net excess return over volatility). Costs are applied after the fact and do not enter the optimisation, as in the replication. Expectations at the base cost: P3a, the long-only portfolio's net excess return exceeds the unconstrained portfolio's in at least six of the eight main variants; P3b, the same for the Sharpe ratio net of cost. P3c, arithmetic from the turnover already in the table above: the unconstrained sample portfolio on monthly data costs about 4.6% a year (0.763 times 0.005 times 12) and the long-only portfolios 0.3% to 0.5%.

In [ ]:
# Added after the run (4 October 2026): P1 and P2.
def decade_label(month):
    if month.year < 1990:
        return "1979-07 to 1989-12"
    if month.year >= 2020:
        return f"2020-01 to {eval_months[-1]}"
    return f"{(month.year // 10) * 10}s"

DECADES = ["1979-07 to 1989-12", "1990s", "2000s", "2010s", f"2020-01 to {eval_months[-1]}"]
decade_of = pd.Series([decade_label(m) for m in eval_months], index=eval_months)

vr_rows = []
for month in eval_months:
    row = {"month": month}
    for name, years in (("variance ratio (3y)", C.EXT_COV_DAILY_WINDOW_YEARS), ("variance ratio (10y)", C.COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY)):
        bd = bench_d.loc[CV.daily_window(excess_d, month, years).index]
        row[name] = CV.variance_ratio(bd.to_numpy(), C.COV_SCALE_VR_HORIZON_DAYS)
    vr_rows.append(row)
vr = pd.DataFrame(vr_rows).set_index("month")
scaling = scaling.join(vr)
print("the variance ratio by decade of the window's last month (1 when days are independent), beside the ratio of expectation 4:")
print(scaling.groupby((scaling.index.year // 10) * 10)[["ratio", "variance ratio (3y)", "variance ratio (10y)"]].mean().to_string(float_format=lambda x: f"{x:.3f}"))
print(f"correlation across windows between the in-window variance ratio (3y) and the ratio of expectation 4: {scaling['ratio'].corr(scaling['variance ratio (3y)']):.2f} (both measure the same window, so this is close to 1 by construction)")
print()


def bias_table(df, factor=None):
    f = np.ones(len(df)) if factor is None else factor.loc[df.index].to_numpy()
    out = {}
    for lab in ["all months"] + DECADES:
        mask = np.ones(len(df), bool) if lab == "all months" else (decade_of.loc[df.index] == lab).to_numpy()
        sub, fs = df[mask], f[mask]
        ind = [CV.bias_statistic(np.stack(sub.real_industries)[:, j], np.stack(sub.pred_industries)[:, j] * fs) for j in range(49)]
        act = [CV.bias_statistic(np.stack(sub.real_active)[:, j], np.stack(sub.pred_active)[:, j] * fs) for j in range(A.shape[0])]
        out[lab] = {"months": int(mask.sum()), "benchmark": CV.bias_statistic(sub.real_bench, sub.pred_bench * fs),
                    "1/N": CV.bias_statistic(sub.real_ew, sub.pred_ew * fs), "industries, mean": float(np.mean(ind)), "active, mean": float(np.mean(act))}
    return pd.DataFrame(out).T


by_decade_bias = pd.concat({
    "sample, monthly_120": bias_table(frames[("sample", "monthly_120")]),
    "sample, daily_3y, 21 rule": bias_table(frames[("sample", "daily_3y")]),
    "sample, daily_3y, corrected (3y)": bias_table(frames[("sample", "daily_3y")], vr["variance ratio (3y)"]),
    "sample, daily_3y, corrected (10y)": bias_table(frames[("sample", "daily_3y")], vr["variance ratio (10y)"]),
}, names=["forecast", "period"])
print("P2, bias statistics by period (1 is calibrated; above 1, risk under-forecast):")
print(by_decade_bias.to_string(float_format=lambda x: f"{x:.3f}"))
print()

rule = by_decade_bias.loc["sample, daily_3y, 21 rule", "benchmark"]
corr3 = by_decade_bias.loc["sample, daily_3y, corrected (3y)", "benchmark"]
closer = [d for d in DECADES if abs(corr3[d] - 1) < abs(rule[d] - 1)]
print(f"P1a: the corrected benchmark forecast is closer to 1 than the 21 rule in {len(closer)} of {len(DECADES)} decades ({', '.join(closer)}); "
      f"level {C.COV_POST_RUN_DECADES_MIN_CLOSER}: {'MET' if len(closer) >= C.COV_POST_RUN_DECADES_MIN_CLOSER else 'NOT MET'}")
p2a = rule[DECADES[0]] > 1 and rule["2010s"] < 1 and rule[DECADES[0]] > rule["2000s"]
print(f"P2a: under the 21 rule the benchmark's bias statistic is {rule[DECADES[0]]:.3f} in {DECADES[0]} (above 1: {rule[DECADES[0]] > 1}), {rule['2010s']:.3f} in the 2010s (below 1: {rule['2010s'] < 1}), "
      f"{rule['2000s']:.3f} in the 2000s (first decade higher: {rule[DECADES[0]] > rule['2000s']}): {'MET' if p2a else 'NOT MET'}")
print()
others = pd.DataFrame({f"{v[0]}, daily_3y": {"21 rule": bias_table(frames[v]).loc["all months"], "corrected (3y)": bias_table(frames[v], vr["variance ratio (3y)"]).loc["all months"]}
                       for v in MAIN if v[1] == "daily_3y"}).T
others = pd.concat({k: pd.DataFrame(v.tolist(), index=v.index) for k, v in others.items()}, names=["scale", "estimator"]).swaplevel().sort_index()
print("P1b, all months, the four daily estimators under the 21 rule and corrected (3y):")
print(others.drop(columns="months").to_string(float_format=lambda x: f"{x:.3f}"))

In [ ]:
# Added after the run (4 October 2026): P3, trading costs.
cost_rows = []
for v in MAIN:
    df = frames[v]
    for port, label in (("lo", "long-only"), ("un", "unconstrained")):
        ret, to = df[f"gmv_{port}_ret"], df[f"gmv_{port}_turnover"].mean()
        gross, vol = 12 * ret.mean(), CV.annualised_vol(ret)
        row = {"estimator": v[0], "data": v[1], "portfolio": label, "turnover a month": to, "gross excess return % a year": 100 * gross, "volatility % a year": 100 * vol}
        for c in C.EXT_COST_LEVELS:
            bp = int(round(c * 1e4))
            cost = 12 * to * c
            row[f"cost % a year at {bp} bp"] = 100 * cost
            row[f"net excess return % at {bp} bp"] = 100 * (gross - cost)
            row[f"Sharpe net at {bp} bp"] = (gross - cost) / vol
        cost_rows.append(row)
costs = pd.DataFrame(cost_rows).set_index(["estimator", "data", "portfolio"])
print("P3, the minimum-variance portfolios with trading costs (eight main variants):")
print(costs.to_string(float_format=lambda x: f"{x:.2f}"))
print()
base = int(round(C.EXT_COST_LEVELS[0] * 1e4))
lo_c, un_c = costs.xs("long-only", level="portfolio"), costs.xs("unconstrained", level="portfolio")
n_ret = int((lo_c[f"net excess return % at {base} bp"] > un_c[f"net excess return % at {base} bp"]).sum())
n_sr = int((lo_c[f"Sharpe net at {base} bp"] > un_c[f"Sharpe net at {base} bp"]).sum())
print(f"P3a: at {base} bp the long-only portfolio's net excess return exceeds the unconstrained portfolio's in {n_ret} of {len(MAIN)} variants (level {C.COV_POST_RUN_COST_MIN_VARIANTS}): {'MET' if n_ret >= C.COV_POST_RUN_COST_MIN_VARIANTS else 'NOT MET'}")
print(f"P3b: the same for the Sharpe ratio net of cost: {n_sr} of {len(MAIN)}: {'MET' if n_sr >= C.COV_POST_RUN_COST_MIN_VARIANTS else 'NOT MET'}")
print(f"P3c: the unconstrained sample portfolio on monthly data costs {un_c.loc[('sample', 'monthly_120'), f'cost % a year at {base} bp']:.1f}% a year at {base} bp; "
      f"the long-only portfolios cost {lo_c[f'cost % a year at {base} bp'].min():.1f}% to {lo_c[f'cost % a year at {base} bp'].max():.1f}%")

## Save the summary for the later notebooks

The summary table, the scaling check with the variance ratios, the bias statistics by period and the cost table are written to `outputs/`, with the vintage in the file name, as a record; notebooks 11 to 13 rebuild what they need from the modules.

In [ ]:
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
vintage = provenance[0]["vintage_line"].split("the ")[1].split(" ")[0]
table.to_csv(f"{C.OUTPUT_DIR}/covariance_estimators_{vintage}.csv", float_format="%.6f")
scaling.to_csv(f"{C.OUTPUT_DIR}/daily_to_monthly_scaling_{vintage}.csv", float_format="%.6f")
by_decade_bias.to_csv(f"{C.OUTPUT_DIR}/bias_by_period_{vintage}.csv", float_format="%.6f")
costs.to_csv(f"{C.OUTPUT_DIR}/minimum_variance_costs_{vintage}.csv", float_format="%.6f")
print("written:", sorted(f for f in os.listdir(C.OUTPUT_DIR) if f.startswith(("covariance", "daily_to", "bias_by", "minimum_variance"))))

## What this notebook established, and what could be wrong

| Expectation | Result | Verdict |
|---|---|---|
| 1. Count first | 686 months and 14,414 trading days from 1969-07, one missing day left out, 566 evaluation months for every estimator | holds |
| 2. Daily data beat monthly data for the sample estimator | unconstrained minimum-variance volatility 11.9% against 14.1%; bias statistics 1.028 against 1.023 for the benchmark and 1.060 against 1.060 for the active portfolios | 1 of 3 parts |
| 3. The finding of Dom, Howard, Jansen and Lohre (2024) | long-only volatility 12.11% to 12.20% across the four daily estimators, a range of 0.09 points against the level of 1; the sample covariance on monthly data has the highest unconstrained volatility of the eight main variants (14.1%), and the two-component sensitivity on daily data is higher (14.3%) | MET; the second part holds for the main variants only |
| 4. 21 times a daily variance is a monthly variance | mean ratio 0.92, inside 0.8 to 1.25; 186 of 566 windows inside; 1.82 for windows ending in the 1970s and 0.62 in the 2000s; correlation 0.81 with the daily autocorrelation | MET on the mean, fails in 380 windows |
| 5. Sensitivities, shrinkage intensity, condition numbers | exponential weighting gives the lowest unconstrained volatility (11.7%); median intensity 0.07 on monthly data and 0.01 on daily; condition numbers 1,859 for the sample covariance against 349 after shrinkage | reported |
| P1a. The variance-ratio correction brings the benchmark's bias statistic closer to 1 in at least 4 of 5 decades | 4 of 5 (every decade except the 1990s) | MET |
| P1b. The correction for 1/N, the industries and the active portfolios | 1.16, 1.16 and 1.13 under the correction against 1.12, 1.10 and 1.06 under the 21 rule | reported |
| P2a. Under the 21 rule the benchmark's bias statistic is above 1 in the first decade, below 1 in the 2010s, and higher in the first decade than in the 2000s | 1.34, 0.78 and 0.87 | MET |
| P3a. At 50 basis points the long-only portfolio's net excess return exceeds the unconstrained portfolio's in at least 6 of 8 variants | 8 of 8 | MET |
| P3b. The same for the Sharpe ratio net of cost | 8 of 8 | MET |
| P3c. The cost of the unconstrained sample portfolio on monthly data | 4.6% a year at 50 basis points; the long-only portfolios 0.3% to 0.5% | as computed |

Expectation 1 holds: 686 months and 14,414 trading days from 1969-07, one missing day left out, factors on every day of every window, 566 evaluation months from 1979-07 to 2026-08 for every estimator. A three-year daily window holds about 752 days, 36,848 observations for the 1,225 numbers, against 5,880 in the monthly window.

Expectation 2, the frequency question, holds in one of its three parts. Where the covariance is inverted, daily data win clearly: the unconstrained minimum-variance portfolio built from the sample covariance realises 14.1% a year of volatility on monthly data and 11.9% on daily data, with a turnover of 0.76 a month against 0.40 and a condition number of 1,859 against 691. Where the covariance is only read, as a forecast of a portfolio's risk, daily data do no better: the bias statistic for the benchmark is 1.028 on daily data against 1.023 on monthly, and for the active portfolios 1.060 against 1.060. Expectation 4 says why, and the two halves of the result are consistent with each other: the minimum-variance weights do not change when every entry of the covariance matrix is multiplied by the same number (the weights are Sigma^-1 times a vector of ones, scaled to sum to one, and the scale cancels), so the portfolio test cannot see an error in the daily-to-monthly scale, while a forecast of variance is that scale. Daily data improved what the portfolio test measures, the pattern of the matrix, and left what the forecast test measures, its level over a month, to the scaling step.

Expectation 3, the finding of Dom, Howard, Jansen and Lohre (2024), holds. With the long-only constraint, the minimum-variance portfolios of the four estimators on daily data realise between 12.11% and 12.20% a year, a range of 0.09 points against the level of 1 point, and on monthly data between 12.15% and 12.32%: under the constraint, the choice of estimator and even the choice of data frequency make no visible difference. The constraint does the estimator's work, because forbidding negative weights removes the long-short positions in which the noisiest directions of the covariance are exploited; the long-only portfolio holds 8 to 10 of the 49 industries whatever the estimate. Without the constraint the estimate matters: the sample covariance on monthly data realises 14.1%, the highest of the eight main variants, against 12.1% for Ledoit-Wolf shrinkage, 12.8% for the factor model and 12.7% for the principal-component model on the same data. Among all fifteen variants, including the sensitivities, the highest is the two-component model on daily data at 14.3%, so the second part of the expectation holds for the eight main variants and fails when the sensitivities are counted. On daily data the order reverses: the sample covariance (11.9%) and shrinkage (11.9%, with a median intensity of only 0.012) are the best, and the factor model (13.2%) and the component model (13.5%) are the worst. With 756 observations the sample covariance is no longer noisy enough for shrinkage to have anything to correct, and the structured estimators pay for their assumption that whatever two industries share beyond the factors is zero.

Expectation 4, the scaling check, is met on average and fails in most windows. The mean ratio of the benchmark's monthly variance to 21 times its daily variance is 0.92, inside 0.8 to 1.25; the median is 0.83, and only 186 of the 566 windows lie inside the range: 108 above 1.25 and 272 below 0.8. The ratio moves with the decade: 1.82 for windows ending in the 1970s, 1.41 in the 1980s, 1.06 in the 1990s, 0.62 in the 2000s, 0.64 in the 2010s and 0.77 in the 2020s. The autocorrelation of the benchmark's daily return explains it, with a correlation of 0.81 across windows: a day's return predicted the next day's positively in the 1970s (autocorrelation 0.21), so monthly variance was larger than 21 daily variances, and negatively since 2000 (-0.03 to -0.12), so it was smaller. The rule that variance grows in proportion to time assumes that days are independent, and for these portfolios they were not, in one direction for thirty years and in the other for twenty-five. This is why the daily-based forecasts of risk are no better calibrated than the monthly ones: on 1/N and on the single industries, where the effect is strongest, the daily-based bias statistics are 1.10 to 1.13 against 1.02 to 1.04 for the monthly-based ones, an under-forecast of risk of about ten percent that the 1970s and 1980s produce. P1 tests the remedy.

Expectation 5, the sensitivities. One year of daily data gives the same unconstrained volatility as three (11.9%) with three times the turnover (1.18 a month against 0.40); five years gives 11.9% with 0.26. Exponential weighting with a one-year half-life over five years gives the lowest unconstrained volatility of all variants, 11.7%, and the best-calibrated forecast for the benchmark, a bias statistic of 0.998, at the cost of a turnover of 0.48: weighting recent days more is worth more than any choice of structure, which is what Dom, Howard, Jansen and Lohre found. The constant-correlation shrinkage target shrinks much harder (median intensity 0.27 on monthly data against 0.07 for the scaled identity) and does the same or slightly worse. The two-component model does worse than the five-component one without the constraint (13.3% against 12.7% on monthly data, 14.3% against 13.5% on daily) and the same with it. The condition numbers say what shrinkage does to the inverse: the sample covariance on monthly data has a median condition number of 1,859 and Ledoit-Wolf shrinkage 349, so the inverse of the sample covariance amplifies errors about five times more.

**The tests added after the run.**

P1, the scale correction. The variance ratio measured inside each three-year window follows the ratio of expectation 4 (correlation 0.91 across windows) and moves with the decade: 1.52 for windows ending in the 1970s, 0.69 for windows ending in the 2010s. Scaling the daily covariance by that ratio times 21, in place of 21, brings the benchmark's bias statistic closer to 1 in four of the five decades (1.14 against 1.34 in 1979-07 to 1989-12, 1.07 against 0.87 in the 2000s, 0.95 against 0.78 in the 2010s, 1.02 against 0.89 from 2020) and further from it in the 1990s (1.11 against 1.08), the one decade where the 21 rule was already close. Over all 566 months at once the corrected statistic is 1.07 against 1.03 under the rule, and the two numbers do not say the same thing: under the rule the under-forecasts of the first decade and the over-forecasts of the 2000s and 2010s cancel inside one statistic; under the correction every decade but the 2010s sits between 1.02 and 1.14 and nothing cancels. The ten-year variance ratio does better than the three-year one in every decade: 1.09, 1.04, 0.97, 0.95 and 1.03, a range of 0.14, against 0.78 to 1.34 under the 21 rule and 0.84 to 1.20 for the forecast from 120 monthly returns. A three-year window gives a noisy ratio (its standard error is about 0.19 when days are independent) and the autocorrelation moves over decades, so ten years of it is the better estimate. This reopens the forecast half of expectation 2 after the fact: with a scale that accounts for the autocorrelation, three years of daily data forecast the benchmark's monthly risk more evenly across the decades than 120 monthly returns do. The correction does not help 1/N, the industries or the active portfolios (1.16, 1.16 and 1.13 against 1.12, 1.10 and 1.06), because the ratio is the benchmark's: 1/N and the single industries had stronger positive autocorrelation than the cap-weighted benchmark (under the 21 rule their bias statistics reach 1.46 and 1.36 in the first decade against the benchmark's 1.34), so a correction fitted to the benchmark fits them less. The other three daily estimators move with the sample estimator, within 0.02 of it.

P2, the bias statistics by decade. The 21 rule's forecast on daily data under-forecast the benchmark's risk in the first decade (1.34) and over-forecast it in the 2000s (0.87) and the 2010s (0.78), as the ratios of expectation 4 implied. The forecast from 120 monthly returns swings for a different reason: it ranges from 0.84 in the 2010s, when its window still carried 2008 and the market was calm, to 1.20 from 2020, when the fall of March 2020 met a window built on calm years. A 120-month window adapts to a change in the level of volatility over ten years; a three-year daily window adapts within three.

P3, trading costs. The unconstrained portfolios earn less before costs than the long-only ones in every one of the eight variants (gross excess returns of 4.0% to 7.3% a year against 7.5% to 8.0%) and trade three to eleven times as much (0.21 to 0.76 a month against 0.05 to 0.09). At 50 basis points per unit of turnover their net excess returns are 2.3% to 5.7% a year against 7.1% to 7.5%, and their Sharpe ratios net of cost 0.17 to 0.45 against 0.58 to 0.62. The unconstrained sample portfolio on monthly data pays 4.6% a year in costs, two thirds of its gross return; at 100 basis points its net return is negative. Only two unconstrained portfolios realise less volatility than their long-only counterpart, the sample covariance and shrinkage on daily data, by 0.24 and 0.35 points of volatility, and they pay 1.9 and 1.6 points of return a year more in costs for it. This is expectation 3 seen from the cost side, and it is one reason why the extension's portfolio is long-only with a turnover cap, a limit on the fraction of the portfolio that may change hands in a month.

**What this notebook does not settle.**

- The extension's portfolio is long-only with bounds on active weights and turnover, and expectation 3 says that under such constraints the estimators realise the same risk to within a tenth of a percentage point. Notebook 12 measures whether that holds for tracking error against the benchmark, which is the extension's question, and the answer may be that the choice of estimator does not matter there either.
- The scale correction is one number taken from the benchmark. A portfolio whose autocorrelation differs from the benchmark's, 1/N or a single industry, keeps part of its miss, and a correction per portfolio would need each portfolio's own variance ratio, or the matrix of autocovariances, which three years of daily data cannot estimate with useful precision. Notebook 12 reports its forecast tracking error under the 21 rule and under the ten-year variance ratio.
- The cost figures use one proportional cost per unit of turnover for every industry and every month, as the replication does; costs differ across industries and have fallen over the decades. The cost differences between the portfolios are arithmetic on their turnover and are not noisy; the differences in gross return are, because a mean return over 566 months has a standard error of about 1.8 percentage points a year, so the gross return gaps of 0.4 to 3.6 points between long-only and unconstrained portfolios are mostly within noise, and P3a and P3b rest on the costs.
- P1 to P3 were designed after the first run, with their expectations fixed before their code but after the outcomes of expectations 1 to 5 were known. That is weaker than fixing them before any run: P2a in particular restates what the ratio table of expectation 4 already implied, and P1a was written knowing that the 21 rule missed by a factor of 1.8 and 0.6 in the decades where a correction has most to gain.